## Database Setup and Connection Testing

This notebook sets up the IBM i database for 3SAT problems and tests the Mapepire connection.

**Author:** Quantum-IBMi-Docker Project  
**Based on:** Jack Woehr's COMMON 2021 Presentation

### Import Required Libraries

In [1]:
import getpass
from mapepire_python.client.sql_job import SQLJob
from mapepire_python.data_types import DaemonServer
from IPython.display import Markdown, display

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


### Connect to IBM i via Mapepire

In [2]:
# Get connection details from user
host = input("Enter IBM i hostname/IP: ").strip()
username = input("Enter username: ").strip()
password = getpass.getpass("Enter password: ")
port_input = input("Enter port (press Enter for default 8076): ").strip()
port = int(port_input) if port_input else 8076

# Create daemon server configuration
daemon_server = DaemonServer(
    host=host,
    port=port,
    user=username,
    password=password
)

# Create SQL job and connect
sql_job = SQLJob()
sql_job.connect(daemon_server)

display(Markdown("**✓ Connected to IBM i database**"))

Enter IBM i hostname/IP:  idevphp.idevcloud.com
Enter username:  Jgorzins
Enter password:  ········
Enter port (press Enter for default 8076):  


**✓ Connected to IBM i database**

### Test Connection with Simple Query

In [3]:
# Test query to verify connection
print("Testing query: SELECT CURRENT_USER...")
query = sql_job.query("SELECT CURRENT_USER FROM SYSIBM.SYSDUMMY1")
result = query.run()

if result.get("success") and result.get("data"):
    first_row = result['data'][0]
    current_user = list(first_row.values())[0]
    print(f"✓ Current user: {current_user}")
else:
    print(f"✗ Query failed: {result.get('error', 'Unknown error')}")

Testing query: SELECT CURRENT_USER...
✓ Current user: JGORZINS


### Set Current Schema to JESSEG

In [4]:
# Set the current schema to JESSEG
try:
    query = sql_job.query("SET SCHEMA JESSEG")
    result = query.run()
    if result.get("success"):
        print("✓ Schema set to JESSEG successfully")
    else:
        print(f"Note: {result.get('error', 'Could not set schema')}")
except Exception as e:
    print(f"Note: Error setting schema: {e}")

✓ Schema set to JESSEG successfully


### Create Database Schema

In [5]:
# Create JESSEG schema if it doesn't exist
try:
    query = sql_job.query("CREATE SCHEMA JESSEG")
    result = query.run()
    if result.get("success"):
        print("✓ Schema JESSEG created successfully")
    else:
        print(f"Note: {result.get('error', 'Schema might already exist')}")
except Exception as e:
    print(f"Note: Couldn't create schema. Probably already exists. ({e})")

Note: Couldn't create schema. Probably already exists. ({'error': '[SQL0601] JESSEG in *N type *LIB already exists.', 'sql_state': '42710', 'sql_rc': -601})


### Create THREESAT Function with Sample Data

In [6]:
# Create the THREESAT function with embedded sample data
create_function_sql = """
CREATE OR REPLACE FUNCTION JESSEG.THREESAT()
  RETURNS TABLE (c1 INT, c2 INT, c3 INT, c4 INT)
  LANGUAGE SQL
  SPECIFIC JESSEG.THREESAT
  NOT DETERMINISTIC
  NO EXTERNAL ACTION
  RETURN
    VALUES (-1, -2, -3, 0),
           ( 1, -2,  3, 0),
           ( 1,  2, -3, 0),
           ( 1, -2, -3, 0),
           (-1,  2,  3, 0)
"""

try:
    query = sql_job.query(create_function_sql)
    result = query.run()
    if result.get("success"):
        print("✓ Function JESSEG.THREESAT() created successfully")
        print("  Function contains 5 sample 3SAT clauses")
    else:
        print(f"✗ Failed to create function: {result.get('error', 'Unknown error')}")
except Exception as e:
    print(f"✗ Error creating function: {e}")

✓ Function JESSEG.THREESAT() created successfully
  Function contains 5 sample 3SAT clauses


### Verify THREESAT Function

In [7]:
# Query the THREESAT function to verify data
query = sql_job.query("SELECT * FROM TABLE(JESSEG.THREESAT()) AS t")
result = query.run()

if result.get("success") and result.get("data"):
    print(f"✓ Data verification successful")
    print(f"  Found {len(result['data'])} clauses in JESSEG.THREESAT()")
    print("\n3SAT Clauses (format: c1, c2, c3, c4):")
    for i, row in enumerate(result['data'], 1):
        print(f"  Clause {i}: {row}")
else:
    print(f"✗ Failed to verify data: {result.get('error', 'No data found')}")

✓ Data verification successful
  Found 5 clauses in JESSEG.THREESAT()

3SAT Clauses (format: c1, c2, c3, c4):
  Clause 1: {'C1': -1, 'C2': -2, 'C3': -3, 'C4': 0}
  Clause 2: {'C1': 1, 'C2': -2, 'C3': 3, 'C4': 0}
  Clause 3: {'C1': 1, 'C2': 2, 'C3': -3, 'C4': 0}
  Clause 4: {'C1': 1, 'C2': -2, 'C3': -3, 'C4': 0}
  Clause 5: {'C1': -1, 'C2': 2, 'C3': 3, 'C4': 0}


### Close Connection

In [8]:
# Close the SQL job connection
sql_job.close()
print("✓ Connection closed successfully")

✓ Connection closed successfully


### Summary

This notebook has:
1.  Connected to IBM i via Mapepire
2.  Set the current schema to JESSEG
3.  Created the JESSEG schema
4.  Created the THREESAT() function with sample 3SAT data
5.  Verified the function returns correct data

You can now run the `3sat.ipynb` notebook to solve 3SAT problems using Grover's algorithm!